In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
BRONZE_PATH = "abfss://bronze@pravdatalake.dfs.core.windows.net"
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"
SILVER_TABLE_PATH = f"{SILVER_PATH}/sales_table"
SILVER_TABLE_NAME = "vehicle_sales.silver.sales_table"

In [0]:
sales_df = spark.read.format("csv")\
    .option("header", True)\
    .load(f"{BRONZE_PATH}/Sales_table")

In [0]:
sales_df.display()

In [0]:
print(f"bronze row count: {sales_df.count()}")

In [0]:
# Sales_table arrives wide - one column per year (2001-2020).
# Unpivot to long format: one row per Genmodel_ID + Year.
year_columns = [str(y) for y in range(2001, 2021)]
present_year_columns = [c for c in year_columns if c in sales_df.columns]
stack_expr = ", ".join([f"'{y}', `{y}`" for y in present_year_columns])
n = len(present_year_columns)
 
unpivoted_df = sales_df.selectExpr(
    "Maker", "Genmodel", "Genmodel_ID",
    f"stack({n}, {stack_expr}) as (Year, Units_Sold)"
)

In [0]:
silver_sales = (
    unpivoted_df
    .withColumn("Genmodel_ID", trim(col("Genmodel_ID")))
    .withColumn("Maker", trim(initcap(col("Maker"))))
    .withColumn("Genmodel", trim(col("Genmodel")))
    .withColumn("Year", expr("try_cast(regexp_replace(Year, '[^0-9]', '') as int)"))
    .withColumn("Units_Sold", expr("try_cast(regexp_replace(Units_Sold, '[^0-9]', '') as int)"))
    .filter(col("Genmodel_ID").isNotNull())
    .filter(col("Year").isNotNull())
    .filter((col("Units_Sold").isNull()) | (col("Units_Sold") >= 0))
    .dropDuplicates(["Genmodel_ID", "Year"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)

In [0]:
silver_sales.display()

####Data Quality checks

In [0]:
row_count = silver_sales.count()

In [0]:
null_key_count = silver_sales.filter(col("Genmodel_ID").isNull() | col("Year").isNull()).count()

In [0]:
duplicate_key_count = silver_sales.groupBy("Genmodel_ID", "Year").count().filter("count > 1").count()

In [0]:
negative_units_count = silver_sales.filter(col("Units_Sold") < 0).count()

In [0]:
print(f"silver row count: {row_count}")
print(f"null key count: {null_key_count}")
print(f"duplicate key count: {duplicate_key_count}")
print(f"negative Units_Sold count: {negative_units_count}")

In [0]:
assert null_key_count == 0, "Genmodel_ID/Year should never be null in silver_sales"
assert duplicate_key_count == 0, "(Genmodel_ID, Year) should be unique in silver_sales"
assert negative_units_count == 0, "Units_Sold should never be negative"

In [0]:
if DeltaTable.isDeltaTable(spark, SILVER_TABLE_PATH):
 
    silver_table = DeltaTable.forPath(spark, SILVER_TABLE_PATH)
 
    (silver_table.alias("t")
        .merge(silver_sales.alias("s"), "t.Genmodel_ID = s.Genmodel_ID AND t.Year = s.Year")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
 
else:
 
    silver_sales.write \
        .format("delta") \
        .mode("overwrite") \
        .save(SILVER_TABLE_PATH)
 


In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE_NAME}
    USING DELTA
    LOCATION '{SILVER_TABLE_PATH}'
""")

In [0]:
spark.sql(f"OPTIMIZE {SILVER_TABLE_NAME} ZORDER BY (Genmodel_ID)")